In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Configuración de aleatoriedad para reproducibilidad
np.random.seed(42)
random.seed(42)

NUM_REGISTROS = 25000

print(f"Generando dataset con {NUM_REGISTROS} registros...")

# 1. Generación de IDs y Datos Demográficos Básicos
id_pacientes = [f"PAC-{i:05d}" for i in range(1, NUM_REGISTROS + 1)]
sexos = np.random.choice(['M', 'F'], size=NUM_REGISTROS, p=[0.49, 0.51])

# Edades concentradas en adultos (donde el riesgo de infarto es más dinámico)
edades = np.random.normal(loc=52, scale=15, size=NUM_REGISTROS).astype(int)
edades = np.clip(edades, 18, 90)

# 2. Fechas de Nacimiento coherentes con la edad
def generar_fecha_nacimiento(edad):
    año_nac = 2026 - edad
    mes = random.randint(1, 12)
    # Manejo simple de días del mes
    if mes == 2:
        dia = random.randint(1, 28)
    elif mes in [4, 6, 9, 11]:
        dia = random.randint(1, 30)
    else:
        dia = random.randint(1, 31)
    return f"{año_nac:04d}-{mes:02d}-{dia:02d}"

fechas_nacimiento = [generar_fecha_nacimiento(e) for e in edades]

# 3. Nombres y Apellidos Simulados
nombres_m = ['Juan', 'Pedro', 'Carlos', 'Luis', 'Jose', 'Antonio', 'Miguel', 'Francisco', 'Manuel', 'Alejandro']
nombres_f = ['Maria', 'Ana', 'Luisa', 'Elena', 'Sofia', 'Carmen', 'Laura', 'Patricia', 'Isabel', 'Guadalupe']
apellidos = ['Martinez', 'Rodriguez', 'Lopez', 'Hernandez', 'Gonzalez', 'Perez', 'Sanchez', 'Ramirez', 'Cruz', 'Gomez', 'Flores', 'Morales', 'Vazquez', 'Reyes', 'Jimenez', 'Torres']

nombres = [random.choice(nombres_m) if s == 'M' else random.choice(nombres_f) for s in sexos]
ap_paternos = [random.choice(apellidos) for _ in range(NUM_REGISTROS)]
ap_maternos = [random.choice(apellidos) for _ in range(NUM_REGISTROS)]

# 4. CURP Ficticia (Primeras letras de apellidos/nombre + año + sexo)
def generar_curp(n, ap, am, f_nac, s):
    pad = (ap[:2] + am[:1] + n[:1]).upper()
    aa = f_nac[2:4]
    mm = f_nac[5:7]
    dd = f_nac[8:10]
    h_m = 'H' if s == 'M' else 'M'
    rand_suffix = "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=5))
    return f"{pad}{aa}{mm}{dd}{h_m}PL{rand_suffix}"

curps = [generar_curp(nombres[i], ap_paternos[i], ap_maternos[i], fechas_nacimiento[i], sexos[i]) for i in range(NUM_REGISTROS)]

# 5. Geografía: Asegurar alta concentración en Xicotepec de Juárez (Puebla)
# Para cumplir la práctica requerimos que haya una buena muestra de Xicotepec.
municipios_lista = ['Xicotepec de Juárez', 'Huauchinango', 'Zacatlán', 'Puebla', 'Teziutlán', 'Ajalpan']
pesos_municipios = [0.40, 0.20, 0.15, 0.10, 0.10, 0.05] # 40% de los datos serán de Xicotepec

municipios = np.random.choice(municipios_lista, size=NUM_REGISTROS, p=pesos_municipios)
regiones = []
latitudes = []
longitudes = []

for muni in municipios:
    if muni == 'Xicotepec de Juárez':
        regiones.append('Sierra Norte')
        # Coordenadas reales aproximadas distribuidas en el municipio
        latitudes.append(round(random.uniform(20.2500, 20.3200), 5))
        longitudes.append(round(random.uniform(-98.0000, -97.9200), 5))
    elif muni == 'Huauchinango':
        regiones.append('Sierra Norte')
        latitudes.append(round(random.uniform(20.1500, 20.2000), 5))
        longitudes.append(round(random.uniform(-98.1000, -98.0000), 5))
    else:
        regiones.append('Otras Regiones')
        latitudes.append(round(random.uniform(18.5000, 19.5000), 5))
        longitudes.append(round(random.uniform(-98.5000, -97.5000), 5))

# 6. Variables Físicas y Clínicas
estaturas = np.round(np.random.normal(loc=1.65, scale=0.08, size=NUM_REGISTROS), 2)
# El peso estará correlacionado ligeramente con la estatura
pesos = np.round((estaturas ** 2) * np.random.normal(loc=26, scale=4, size=NUM_REGISTROS), 1)

# Calcular IMC real basado en peso y estatura
imc = np.round(pesos / (estaturas ** 2), 1)

# Variables clínicas basales
fumador = np.random.choice([0, 1], size=NUM_REGISTROS, p=[0.75, 0.25])
actividad_fisica = np.random.choice([0, 1], size=NUM_REGISTROS, p=[0.60, 0.40]) # 1 = Activo, 0 = Sedentario
antecedente_familiar = np.random.choice([0, 1], size=NUM_REGISTROS, p=[0.70, 0.30])

# Signos vitales influenciados ligeramente por la edad y el IMC
presion_sistolica = np.random.normal(loc=115 + (edades*0.2) + (imc*0.3), scale=10, size=NUM_REGISTROS).astype(int)
presion_diastolica = np.random.normal(loc=75 + (edades*0.1) + (imc*0.2), scale=7, size=NUM_REGISTROS).astype(int)
glucosa = np.random.normal(loc=85 + (imc*0.5), scale=25, size=NUM_REGISTROS).astype(int)
colesterol = np.random.normal(loc=180 + (edades*0.3), scale=35, size=NUM_REGISTROS).astype(int)
frecuencia_cardiaca = np.random.normal(loc=72, scale=8, size=NUM_REGISTROS).astype(int)

# Diagnósticos derivados de los umbrales clínicos usuales
diabetes = np.where(glucosa > 126, 1, np.random.choice([0, 1], size=NUM_REGISTROS, p=[0.95, 0.05]))
hipertension = np.where((presion_sistolica > 140) | (presion_diastolica > 90), 1, np.random.choice([0, 1], size=NUM_REGISTROS, p=[0.93, 0.07]))

# 7. Definición Matemática/Clínica de la Variable Objetivo: Riesgo_Infarto
# Creamos un score de riesgo interno para luego clasificar en Bajo, Medio, Alto, Crítico.
# Ojo: No incluyas este Score en tus variables X del modelo (evita fuga de información).
score_riesgo = (
    (edades * 0.4) + 
    (hipertension * 15) + 
    (diabetes * 15) + 
    (fumador * 12) + 
    ((imc > 30).astype(int) * 10) + 
    (antecedente_familiar * 10) - 
    (actividad_fisica * 8) +
    ((presion_sistolica - 120) * 0.2) + 
    ((colesterol - 200) * 0.05)
)

# Clasificación basada en cuartiles/umbrales del score
riesgo_infarto = []
for score in score_riesgo:
    if score < 25:
        riesgo_infarto.append('Bajo')
    elif score < 45:
        riesgo_infarto.append('Medio')
    elif score < 65:
        riesgo_infarto.append('Alto')
    else:
        riesgo_infarto.append('Crítico')

# Crear el DataFrame
df = pd.DataFrame({
    'ID_Paciente': id_pacientes,
    'CURP': curps,
    'Nombre': nombres,
    'Apellido_Paterno': ap_paternos,
    'Apellido_Materno': ap_maternos,
    'Fecha_Nacimiento': fechas_nacimiento,
    'Edad': edades,
    'Sexo': sexos,
    'Estatura_m': estaturas,
    'Peso_kg': pesos,
    'IMC': imc,
    'Presion_Sistolica': presion_sistolica,
    'Presion_Diastolica': presion_diastolica,
    'Glucosa': glucosa,
    'Colesterol': colesterol,
    'Frecuencia_Cardiaca': frecuencia_cardiaca,
    'Diabetes': diabetes,
    'Hipertension': hipertension,
    'Fumador': fumador,
    'Actividad_Fisica': actividad_fisica,
    'Antecedente_Familiar': antecedente_familiar,
    'Municipio': municipios,
    'Region': regiones,
    'Latitud': latitudes,
    'Longitud': longitudes,
    'Riesgo_Infarto': riesgo_infarto  # Esta será tu columna objetivo ('y')
})

# Guardar en CSV
nombre_archivo = "dataset_pacientes_riesgo.csv"
df.to_csv(nombre_archivo, index=False, encoding='utf-8')

print(f"¡Éxito! Archivo '{nombre_archivo}' generado correctamente.")
print(df['Municipio'].value_counts())
print("\nDistribución del riesgo de infarto generado:")
print(df['Riesgo_Infarto'].value_counts())

Generando dataset con 25000 registros...
¡Éxito! Archivo 'dataset_pacientes_riesgo.csv' generado correctamente.
Municipio
Xicotepec de Juárez    9918
Huauchinango           5069
Zacatlán               3727
Puebla                 2589
Teziutlán              2548
Ajalpan                1149
Name: count, dtype: int64

Distribución del riesgo de infarto generado:
Riesgo_Infarto
Medio      11148
Bajo        6618
Alto        6213
Crítico     1021
Name: count, dtype: int64
